# MTT 的采样优化与 SRe2L

本章我们先说 MTT 的另一个优化，DATM。其次是另一种数据集蒸馏范式，SRe2L。

# Difficulty-Aligned Trajectory Matching

MTT 有一件事似乎做的太简单了：我们从专家轨迹中采样起点 $\theta_t^*$ 并且以此加上 $M$ 步推到终点 $\theta_{t+M}^*$，这个采样是简单的均匀采样。作者没有揭示使用均匀采样的原因，实际上这也许遵从了一个思想，那就是所有阶段的专家轨迹应该被平等对待。

但是是否有可能，这个采样非常重要？这就是 DATM 的出发点。我们指出，这个采样方式不仅非常重要，还应该根据合成数据集大小来选择。假设我们的合成数据集被限制在极低的量级内，我们应该更多采样早期专家轨迹，因为早期专家轨迹中模型正在学习普遍的常见的图案，这指导合成数据集给出更加普遍的梯度；假设我们的合成数据集被放宽到更加高的量级，我们应该加入后期专家轨迹的采样，因为后期专家轨迹中模型正在钻研颗粒度更细致的图案，合成数据集应该适当给出一些更罕见的梯度。

我们详细说。原文是 https://arxiv.org/pdf/2310.05773 Towards Lossless Dataset Distillation via Difficulty-Aligned Trajectory Matching。

## 基本逻辑

首先我们简称 Image per Class 为 IPC。IPC 对于蒸馏数据集而言决定了其应该选择的分布方式，选择更普遍的分布还是更细致的分布。根据 IPC，我们可以调整蒸馏数据集对应的专家轨迹采样时间段。

下面这张图展示了这一思想。我们希望 IPC 极小的数据集主要展示 Easy Patterns，而 IPC 更大的数据集展示一些 Hard Patterns。右侧图表展示了 DATM 在 CIFAR-10 上使用这一思想后的 State-of-the-Art 水平。

<img src="./assets/DATM.png" width="900" height="300">

我们假设完整专家轨迹是
$$\tau^*
=
\{\theta_t^*\mid 0\leq t\leq h\}$$
其中 $h$ 是终点时刻。

我们设置两个边界。$T^{-}$ 是允许采样的最早时间点，$T^{+}$ 是允许采样的最晚时间点。换言之，采样仅仅是
$$T^{-}\leq t\leq T^{+}$$
我们写开完整专家轨迹
$$\tau^*
=
\{
\theta_0^*,\ldots,
\underbrace{
\theta_{T^-}^*,\ldots,\theta_{T^+}^*
}_{\text{Sample}},
\ldots,\theta_h^*
\}$$

因此还可以引出一个技巧，被称为 easy-to-hard curriculum。实际上核心思想就是让采样时间点上届 $T^{+}$ 一开始保持在较低值，随着迭代次数增加逐渐增大。这样可以保证数据集蒸馏初期学习 Easy Patterns 而后期学习 Hard Patterns。写开是
$$T_{\mathrm{cur}}
:
T_{\mathrm{initial}}
\longrightarrow
T^{+}$$

最后的，我们来讨论一下 Soft Label 相关内容。TESLA 做了一个简单的 Training-free Soft Label 初始化，但是 DATM 对这里做了更细致的处理。

首先，如果专家模型给出了错误的 Soft Label 怎么办？这实际上非常有可能，因为专家模型也不是完全正确的。一个错误的 Soft Label 会导致数据集被指引向错误的方向蒸馏，这是我们不愿意看到的。

解决办法很简单，我们尽量减少这种可能的发生。我们使用那些专家模型判断正确类别的数据作为合成数据集初始化。换言之，对每个真实样本 $x_i$，先计算专家给出 logits
$$L_i
=
f_{\theta^*}(x_i)$$
其中 $L_i\in\mathbb R^C$，$C$ 是类别总数。

然后 softmax 得到 Soft Label
$$q_i
=
\operatorname{softmax}(L_i)$$

现在我们挑选出专家判断正确的样本
$$\mathcal D_{\mathrm{sub}}
=
\left\{
(x_i,q_i)
\;\middle|\;
(x_i,y_i)\in\mathcal D_{\mathrm{real}},
\;
\arg\max_c L_{i,c}=y_i
\right\}$$

再从 $\mathcal{D}_{sub}$ 中挑选子集得到合成数据集初始化
$$\mathcal D_{\mathrm{syn}}
=
\{(\widetilde x_j,\widetilde q_j)\}_{j=1}^{m}$$
其中 $\widetilde x_j$ 是可学习的合成图像，$\widetilde q_j$ 是可学习的 Soft Label 并且 $m\ll n$。

请注意，Soft Label 也是可以学习的，这和 TESLA 固定标签思想不一样。这种学习来源于我们最终损失会向标签回传一个梯度。

更多的，DATM 甚至决定让学习率可学习。这里的学习率指的是
$$\widehat\theta_{t+i+1}
=
\widehat\theta_{t+i}
-
\alpha
\nabla_{\widehat\theta_{t+i}}
\ell
\left(
\widehat\theta_{t+i},
b_{t+i}
\right)$$
此处的 $\alpha$。

因此对于计算图
$$(\mathcal D_{\mathrm{syn}},\alpha)
\longrightarrow
\widehat\theta_{t+1}
\longrightarrow
\widehat\theta_{t+2}
\longrightarrow
\cdots
\longrightarrow
\widehat\theta_{t+N}
\longrightarrow
\mathcal L_{\mathrm{match}}$$
其回传三个梯度
$$\nabla_{\widetilde x_j}
\mathcal L_{\mathrm{match}},
\qquad
\nabla_{\widetilde q_j}
\mathcal L_{\mathrm{match}},
\qquad
\nabla_{\alpha}
\mathcal L_{\mathrm{match}}$$
我们可以验证一件事，这三个梯度的计算图都是局部的。这意味着可以那就是按照 TESLA 的思想进行大幅度的显存优化。

## 蒸馏算法

我们现在来说说详细的蒸馏算法。

首先是 Soft Label 初始化。这里比较隐晦的是，作者发现初始化选用的专家模型检查点对于最终效果并不起关键作用，因此这里的挑选不会很重要。我们实际上可以直接选择训练最完好的专家节点，记为 $f_{\theta^*}$。现在利用这个专家模型为数据集选择初始 Soft Label 与子集.

对于真实图像 $x_i$，记
$$L_i=f_{\theta^*}(x_i).$$
仅仅保留
$$\arg\max_c L_{i,c}=y_i$$
其中 $y_i$ 是真实标签。所以最终初始化数据集形式是

$$\mathcal D_{\mathrm{sub}}=\left\{(x_i,\operatorname{softmax}(L_i))\mid \arg\max_cL_{i,c}=y_i\right\}.$$

对于每一轮迭代，做以下事。

首先从所有专家轨迹中挑选出一条轨迹 $\tau^*$，再按照我们设置的调度器决定的 $T^{-}, T^{+}$ 来采样专家节点 $\theta_t^*$ 并且推出 $\theta_{t +M}^*$。初始化学生模型 $\hat{\theta}_t = \theta_t^*$。

对于内层循环，我们有设置好的步数上界 $N$。对于每一步 $0 \le i \le N-1$，采样 mini-batch $b_{t+i} \sim \mathcal{D}_{syn}$，然后对学生模型做梯度下降 
$$\hat{\theta}_{t+i+1} = \hat{\theta}_{t+i+1} - \alpha \nabla \ell(\hat{\theta}_{t+i+1}, b_{t+i})$$
做以上梯度下降直到产出 $\hat{\theta}_{t+N}$，最终计算
$$\mathcal L
=
\frac{
\left\|
\hat{\theta}_{t+N}
-
\theta_{t+M}^*
\right\|_2^2
}{
\left\|
\theta_t^*
-
\theta_{t+M}^*
\right\|_2^2
}$$
现在反向传播梯度更新所有被选择的 mini-batch 数据点和其 Soft Label，以及学生模型学习率 $\alpha$。

以上是一轮迭代内部内容。迭代直到完成所有次数。

下面是完整蒸馏算法。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1:} \text{ Pipeline of our method} \\
\hline
\textbf{Input: } \{\tau^*\}\text{: set of expert parameter trajectories. } N\text{: update times of the surrogate network in} \\
\quad \text{each inner optimization. } M\text{: update times between the start and target expert parameters.} \\
\quad T^-, T, T^+\text{: lower, current upper, final upper bound of the sample range of } t. \, \mathcal{D}_{\text{real}}\text{: original} \\
\quad \text{dataset. } I\text{: interval for expanding the sampling range.} \\
\text{Sample a model } f_{\theta^*} \text{ from } \{\tau^*\}. \\
\text{Construct } \mathcal{D}_{\text{sub}} = \left\{ (x_i, \text{softmax}(L_i)) \mid (x_i, y_i) \in \mathcal{D}_{\text{real}} \textbf{ and } \text{argmax}(L_i) == y_i \right\}, \text{ where} \\
\quad L_i = f_{\theta^*}(x_i). \\
\text{Randomly sample data from } \mathcal{D}_{\text{sub}} \text{ to initialize synthetic dataset } \mathcal{D}_{\text{syn}}. \\
\begin{aligned}
& \textbf{for } \text{iteration} \leftarrow 0 \textbf{ to } \text{max\_iteration} \textbf{ do} \\
& \quad \text{Randomly sample an expert training trajectory } \tau^* \in \{\tau^*\} \text{ with } \tau^* = \{\theta_i^*\}_0^n \\
& \quad \text{Select random start timestamp } t, \text{ where } T^- \le t \le T \\
& \quad \text{Sample } \theta_t^*, \, \theta_{t+M}^* \text{ from } \tau^*, \text{ initialize } \hat{\theta}_t = \theta_t^* \\
& \quad \textbf{for } i \leftarrow 0 \textbf{ to } N-1 \textbf{ do} \\
& \quad\quad b_{t+i} \sim \mathcal{D}_{\text{syn}} \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \, \triangleright \text{sample a mini-batch of distilled dataset} \\
& \quad\quad \hat{\theta}_{t+i+1} = \hat{\theta}_{t+i} - \alpha \nabla \ell(\hat{\theta}_{t+i}, b_{t+i}) \quad \,\, \triangleright \text{update surrogate model with CE loss} \\
& \quad \textbf{for end} \\
& \quad \text{Compute matching loss between } \hat{\theta}_{t+N} \text{ and } \theta_{t+M}^* \text{ with the Equation } \mathcal L
=
\frac{
\left\|
\hat{\theta}_{t+N}
-
\theta_{t+M}^*
\right\|_2^2
}{
\left\|
\theta_t^*
-
\theta_{t+M}^*
\right\|_2^2
} \\
& \quad \text{Update } (x_i, \text{softmax}(L_i)) \in \{b\}_t^{t+N-1} \text{ and } \alpha \text{ with respect to the matching loss} \\
& \quad \textbf{if } (\text{iteration} \% I == 0) \textbf{ and } (T < T^+) \textbf{ then} \\
& \quad\quad T = T + 1 \\
& \textbf{for end}
\end{aligned} \\
\textbf{Output: } \text{distilled dataset } \mathcal{D}_{\text{syn}} \text{ and learning rate } \alpha \\
\hline
\end{array}
$$

## DATM 的总结

DATM 和 TESLA 对 MTT 做了很大的补充，使其成为一个实用性更高的方法。

我想提醒一件事，那就是 MTT 方法不同于 GM 和 DM 的一件事是，MTT 中真实数据集 $\mathcal{D}_{real}$ 从未对蒸馏数据集产生过直接的监督，甚至 MTT 在蒸馏过程中不会用到真实数据集本体。MTT 的监督方式是通过专家轨迹间接给出原始真实数据集的信息，这意味着监督式并不直接。

下面我想谈谈 SRe2L，这是另一种数据集蒸馏范式，也遵从了间接监督的思想。原文是 https://arxiv.org/pdf/2306.13092 Squeeze, Recover and Relabel: Dataset Condensation at ImageNet Scale From A New Perspective。

# SRe2L

实际上这个成果严格的名称是 $\text{SR}e^2\text{L}$，但是我们姑且简称为 SRe2L。

## 基本逻辑

给定真实数据集
$$\mathcal T
=
\{(x_1,y_1),\ldots,(x_{|\mathcal T|},y_{|\mathcal T|})\},$$
我们希望构造蒸馏数据集
$$\mathcal C_{\mathrm{syn}}
=
\{(\widetilde x_1,\widetilde y_1),\ldots,
(\widetilde x_{|\mathcal C|},\widetilde y_{|\mathcal C|})\}, \quad |\mathcal C|
\ll
|\mathcal T|.$$\
得到
$$\theta_{\mathcal C_{\mathrm{syn}}}
=
\arg\min_\theta
\mathcal L_{\mathcal C}(\theta).$$
其中
$$\mathcal L_{\mathcal C}(\theta)
=
\mathbb E_{(\widetilde x,\widetilde y)\in\mathcal C_{\mathrm{syn}}}
\left[
\ell\bigl(\phi_\theta(\widetilde x),\widetilde y\bigr)
\right].$$

我们想要一个目标
$$\sup_{(x,y)\sim\mathcal T}
\left|
\ell\bigl(\phi_{\theta_{\mathcal T}}(x),y\bigr)
-
\ell\bigl(\phi_{\theta_{\mathcal C_{\mathrm{syn}}}}(x),y\bigr)
\right|
\leq
\epsilon.$$
这里表示蒸馏数据集训练模型与真实数据集训练模型之间的性能差距。换言之我们需要
$$\arg\min_{
\mathcal C_{\mathrm{syn}},
|\mathcal C|
}
\left[
\sup_{(x,y)\sim\mathcal T}
\left|
\ell\bigl(\phi_{\theta_{\mathcal T}}(x),y\bigr)
-
\ell\bigl(\phi_{\theta_{\mathcal C_{\mathrm{syn}}}}(x),y\bigr)
\right|
\right].$$

我们之前提及的诸多数据集蒸馏方法，比如 GM 与 MTT，实际上遵循一个原则，那就是使用当前合成数据集训练模型之后考察其在真实数据集上表现。这里无可避免涉及到一件事，那就是合成数据集对于最终损失反向传播时需要穿透非常多次更新。即使我们指出 TESLA 等等重排计算图方法可以加大程度减轻这种负担，这还是难以接受的。

换言之，我们希望取缔以下样式的计算图
$$\mathcal C_{\mathrm{syn}}
\longrightarrow
\theta_1
\longrightarrow
\theta_2
\longrightarrow
\cdots
\longrightarrow
\theta_K
\longrightarrow
\mathcal L_{\mathrm{real}}.$$
更多的，这样的更新方式还有一大缺点，那就是实际上固定步数的更新截断了一部分的内层训练。我指的是
$$\theta_K
\neq
\theta_{\mathrm{converged}}.$$
这意味着得到的梯度是存在偏差的。我们并没有得到合成数据集真正给出的梯度方向。


本质地说，以上计算图特点是强调学生模型的概念，我们希望学生模型的优化和合成数据集的优化可以交替进行。但是是否有一种可能，学生模型的优化和合成数据集的优化完全可以解耦？

我们正式开始提出新的方案。

首先是 Squeeze 阶段，我们训练一个教师模型
$$\theta_{\mathcal T}
=
\arg\min_{\theta}
\mathcal L_{\mathcal T}(\theta).$$
这可以被解释。深度网络把原始高维图像映射到越来越抽象的低维表示，因此训练完成后，原始数据中的关键判别信息被编码进网络参数和内部统计量

请注意这里的一个重要细节，原生 ViT 通常使用 LayerNorm，但是此处我们将其改造为 BatchNorm。这显然是一个 Trade-off 性质极强的改造，原因是 BatchNorm 在现代神经网络中常常因为过于依赖训练数据集分布和形式而被 LayerNorm 取代。但是这里我们做这个决定是因为我们非常需要 BatchNorm 给出的 Running mean 与 variance。我们接下来会详细说。

我们的第二个阶段是 Recover。我们固定教师模型 $\phi_{\theta_{\mathcal T}}$，优化合成图像。目标是
$$\min_{\widetilde x_{\mathrm{syn}}} \Big [
\ell\left(
\phi_{\theta_{\mathcal T}}
(\widetilde x_{\mathrm{syn}}),
y
\right)
+
\mathcal R_{\mathrm{reg}} \Big ],$$
其中，我们称分类损失是
$$\mathcal L_{\mathrm{cls}}
=
\ell\left(
\phi_{\theta_{\mathcal T}}(\widetilde x),
y
\right).$$
这指的是我们将合成数据推向教师模型认知中的分类 $y$ 区域。

但是如果仅仅有分类损失其实并不安全。因为教师模型认知中的分类 $y$ 区域很可能包含了很多教师高置信度但人眼不可理解的图案与纹理，因此我们还需要正则项。这里的正则项 $\mathcal R_{\mathrm{reg}}$ 主要分为两部分，$\mathcal R_{\mathrm{prior}}$ 与 $\mathcal R_{\mathrm{BN}}$。我们先说第一部分

$$\mathcal R_{\mathrm{prior}}
(\widetilde x)
=
\alpha_{\mathrm{TV}}
\mathcal R_{\mathrm{TV}}(\widetilde x)
+
\alpha_{\ell_2}
\mathcal R_{\ell_2}(\widetilde x).$$
其中
$$\mathcal R_{\mathrm{TV}}(\widetilde x)
=
\sum_{i,j}
\left[
(\widetilde x_{i,j+1}-\widetilde x_{i,j})^2
+
(\widetilde x_{i+1,j}-\widetilde x_{i,j})^2
\right]^{\beta/2}.$$
这是一个逐点计算像素的损失，其指的是相邻像素之间不可以差距过大，换言之图像需要更平滑。这会抑制尖锐纹理。

其次是 $\ell_2$ 正则项

$$\mathcal R_{\ell_2}(\widetilde x)
=
\|\widetilde x\|_2.$$

这指的是像素数值不可以过于巨大，同样是抑制极端分布。

现在我们可以来说第二部分 BN 损失
$$\mathcal R_{\mathrm{BN}}(\widetilde x)
=
\sum_l
\left\|
\mu_l(\widetilde x)
-
\mathrm{BN}^{\mathrm{RM}}_l
\right\|_2
+
\sum_l
\left\|
\sigma_l^2(\widetilde x)
-
\mathrm{BN}^{\mathrm{RV}}_l
\right\|_2.$$
其中，教师模型第 $l$ 个 BN 层保存的 Running mean 记为 $\mathrm{BN}^{\mathrm{RM}}_l$，Running variance 记为 $\mathrm{BN}^{\mathrm{RV}}_l$。对于当前 Batch，mean 和 variance 记为 
$$\mu_l(\widetilde x),
\qquad
\sigma_l^2(\widetilde x).$$
所以整个损失的含义是，合成图像需要在教师模型各层内部产生与真实训练数据类似的均值和方差。

最终，完整损失是
$$\mathcal L_{\mathrm{recover}}
=
\mathcal L_{\mathrm{cls}}
+
\lambda_{\mathrm{BN}}
\mathcal R_{\mathrm{BN}}
+
\alpha_{\mathrm{TV}}
\mathcal R_{\mathrm{TV}}
+
\alpha_{\ell_2}
\mathcal R_{\ell_2},$$

但是一个重要细节是，SRe2L 并不是直接将合成数据集中数据点 $\tilde{x}$ 进行优化，而是进行一个 Multi-crop Optimization 操作。对于原始合成图像，我们每次随机裁剪其中一个区域，再缩放到原始尺寸，并且将其视为被优化对象输入上述损失函数
$$\widetilde x^{(r)}
=
\operatorname{RandomResizedCrop}(\widetilde x).$$

由于当前损失只依赖被裁剪区域，所以这一轮只有被裁到的区域收到像素梯度。经过很多次随机裁剪后，一张合成数据集中的不同区域都能逐渐被优化。这一操作旨在让一张合成图像的不同局部区域携带不同训练信息，提高单张图像的信息密度。

现在我们来说第三阶段 Relabel。在前两阶段之中，我们已经完成了数据集的蒸馏。但是我们并不认为此时的标签是合理的，原因是一张合成图片很有可能携带了不同种类信息。比如狗的图片中，草地与天空等等要素实际上在猫的图片中也会出现，而这些局部要素恰恰是我们在 Multi-crop Optimization 中所优化的。

所以数据集蒸馏结束之后学生模型训练方式并不是直接对原始蒸馏数据集和配对 one-hot 标签进行，而是每次也对某张合成数据集中照片做随机裁剪缩放之后的 crop 做学习。换言之，蒸馏数据集是
$$\mathcal D_{\mathrm{syn}}
=
\left\{
\widetilde x_1,\widetilde x_2,\ldots,\widetilde x_m
\right\},$$

训练学生模型时，第 $e$ 个 epoch 中第 $j$ 张图先采样一个增强配置 $\omega_{e,j}$，得到
$$\widetilde x_{e,j}^{\mathrm{aug}}
=
\mathcal A_{\omega_{e,j}}
\left(
\widetilde x_j
\right).$$
现在使用教师模型为其生成 Soft Label。注意这里的教师模型和 Squeeze 模型可以不是同一个，我们随后会谈
$$\widetilde y_{e,j}
=
\operatorname{softmax}
\left(
f_{\theta^*}
\left(
\widetilde x_{e,j}^{\mathrm{aug}}
\right)
\right).$$
最终学生模型训练是考虑损失函数
$$\mathcal L_{\mathrm{student}}
=
-
\sum_c
\widetilde y_{e,j,c}
\log
p_{\phi,c}
\left(
\widetilde x_{e,j}^{\mathrm{aug}}
\right).$$

尽管 Relabel 操作看起来是训练学生模型时在线做的，实际上这是提前准备的。我们会在蒸馏数据时提前记录每张图片每个增强配置与其对应的 Soft Label，训练学生模型时直接取出用于训练。

所以实际上最终的训练要素是
$${
\mathcal D_{\mathrm{final}}
=
\left\{
\widetilde x_j,
\left\{
\omega_{e,j},
\widetilde y_{e,j}
\right\}_{e=1}^{E}
\right\}_{j=1}^{m}
}$$

下面这张图详细展示了 Squeeze 和 Recover 和 Relabel 的三个阶段。

<img src="./assets/SRe2L.png" width="900" height="250">

## 讨论

我们来讨论原文中提到的几件事。首先，教师模型越强效果一定越好吗？

实际上不一定。我们称教师模型训练的 epoch 数量为 Squeezing budget。一个非常反直觉的消融实验结果是，当教师模型 Squeezing budget 在某个基础上逐步提升，得到的蒸馏数据集训练出的学生模型识别准确率居然在逐步下降。这意味着教师模型在数据集上更充分的训练并不意味着更适合作为真实数据集蒸馏的载体。

作者对此的解释是，模型训练得越久，其表示越抽象越判别化，也可能越来越不保留可逆的数据细节。这是说，教师模型早期可能保存颜色, 典型形状与局部纹理等等丰富的空间信息，但是训练后期模型可能发现一些与类别强相关的细节与背景空间才是必要的图像判别信息，并且逐步抛弃早期的典型丰富信息。这是某种意义上的过拟合，意味着教师模型成为了成功的图像分拣者而不是一个成功的信息存留者。

更多的，我们发现一些常见的增加教师模型性能的手段反而同样会导致其蒸馏数据集效果变差，比如数据增强。当我们在训练教师模型时增强训练数据集，这确实增加了教师模型对于数据的泛化能力。但是在 Recover 阶段，这反而增加了反演出真实图像的困难程度。

第二件事是，蒸馏数据集的视觉自然性是否重要？传统的图像反演会为图像加上 TV 正则项和 $L2$ 正则项保证图像对于人类视觉可读，但是作者测试这两项效果发现非常微弱甚至会导致性能倒退。这是因为蒸馏数据集的视觉可读性不是根本，我们真正需要的是图像是否包含判别器所需信息。

其次的，我们上述提到的 Multi-crop Optimization 究竟做了什么？某种意义上这也是一种不考虑视觉可读性增加可判别性的手段。比如火山的图片蒸馏之后可以布满火山口而不是仅仅一座火山，鲨鱼图片可以布满鲨鱼鳍与尾巴，这意味着更高的信息密度。

下面这张图展示了不同选择下的蒸馏数据集图案。

<img src="./assets/ablSRe2L.png" width="600" height="520">

第三件事是，Recover 需要多少个迭代次数？我们考察每张图像需要在固定教师模型上优化多少次数。最终的结论是，这个数字确实与最终性能相关，但是达到一定优化次数之后会产生边际效应，换言之饱和。

此处还有一件事，那就是更大的教师模型需要更大的 Recover budget 才能使得学生模型从蒸馏数据集上训练达到同样的性能。这是因为更大的模型拥有更复杂的变换与特征抽象，这使得反演过程更加困难。

第四件事是，Recovery model 与 Relabeling model 如何选择？先说结论，最好是同一个模型。这看上去有些平凡，实际上是避免 Squeeze 和 Recover 之间产生语义差距。

第五件事是，Soft Label 温度如何选择？这里指的是 Soft Label 输出的 logits 是
$$q_c(\tau)=\frac{\exp(z_c/\tau)}{\sum_{k=1}^{C}\exp(z_k/\tau)}.$$
其中 $q_c(\tau)$ 是第 $c$ 类概率，温度 $\tau$ 越大概率分布越平坦。

最后结论有些出乎意料，对于组别 $\tau\in\{1,5,10,15,20\}$，$\tau=20$ 最终学生模型准确率最高。这意味当我们更加强调类间关系而不是单一类内判别，将梯度方向调整到指向更多方向，最终学生模型训练效果会更佳。更多的，这实际上阻止了教师模型过于尖锐但可能错误的判别，某种意义上是一种正则化。

第六件事是，更大容量架构的学生模型最终在蒸馏数据集上训练效果也更好。这看起来很反直觉，因为更大容量模型一般意味着更大的训练数据需求量和更高性能上限。实验证明最终在 $\mathrm{ResNet\text{-}18}$ 作为 Squeeze 与 Recover 模型时，学生模型有如下性能关系
$$\mathrm{ResNet\text{-}101}
\gt
\mathrm{ResNet\text{-}50}
\gt
\mathrm{ResNet\text{-}18}.$$
这表现出了一定的架构泛化性。

但是这个架构泛化性是仅仅限制在 ResNet 系列内部的，对于 ConvNet 与 ViT 模型还是展现出泛化性上的困难。

## 蒸馏算法

我们给出完整的算法。

$$\begin{aligned}
&\mathbf{Algorithm\ 1:}\quad \text{SRe}^{2}\text{L: Squeeze, Recover, and Relabel} \\[2mm]
&\mathbf{Input:}\quad \mathcal D_{\mathrm{real}} = \{(x_i,y_i)\}_{i=1}^{n} \text{: original real dataset;} \\
&\phantom{\mathbf{Input:}\quad} f_{\theta} \text{: teacher network with parameters }\theta; \qquad C \text{: number of classes;} \\
&\phantom{\mathbf{Input:}\quad} m \text{: number of synthetic images;} \qquad K_{\mathrm{sq}} \text{: number of squeezing iterations;} \\
&\phantom{\mathbf{Input:}\quad} K_{\mathrm{rec}} \text{: number of recovery iterations;} \qquad B \text{: recovery batch size;} \\
&\phantom{\mathbf{Input:}\quad} \mathcal A_{\omega} \text{: differentiable augmentation parameterized by }\omega; \\
&\phantom{\mathbf{Input:}\quad} \lambda_{\mathrm{BN}}, \lambda_{\mathrm{TV}}, \lambda_{\ell_2} \text{: regularization coefficients;} \\
&\phantom{\mathbf{Input:}\quad} \tau \text{: soft-label temperature;} \qquad E \text{: number of relabeling or student-training epochs.} \\[3mm]
&\mathbf{Stage\ 1:\ Squeeze} \\
&\text{Initialize the teacher parameters }\theta_0. \\
&\mathbf{for}\quad k\leftarrow 0\quad\mathbf{to}\quad K_{\mathrm{sq}}-1\quad\mathbf{do} \\
&\qquad \text{Sample a real-data mini-batch } \mathcal B_k = \{(x_b,y_b)\}_{b=1}^{B} \sim \mathcal D_{\mathrm{real}}. \\
&\qquad \mathcal L_{\mathrm{real}} = \frac{1}{B} \sum_{b=1}^{B} \operatorname{CE} \left( f_{\theta_k}(x_b), y_b \right). \\
&\qquad \theta_{k+1} = \theta_k - \eta_{\theta} \nabla_{\theta_k} \mathcal L_{\mathrm{real}}. \\
&\mathbf{end\ for} \\
&\text{Set } \theta_{\mathcal T} = \theta_{K_{\mathrm{sq}}} \text{ and freeze the teacher network }f_{\theta_{\mathcal T}}. \\
&\text{Store the running mean and variance of every BN layer:} \\
&\qquad \left\{ \mu_l^{\mathrm{run}}, \left(\sigma_l^2\right)^{\mathrm{run}} \right\}_{l=1}^{L}. \\[3mm]
&\mathbf{Stage\ 2:\ Recover} \\
&\text{Initialize the synthetic images and assign balanced target classes:} \\
&\qquad \widetilde{\mathcal X}^{(0)} = \{ \widetilde x_j^{(0)} \}_{j=1}^{m}, \qquad \widetilde x_j^{(0)} \sim \mathcal N(0,I), \qquad \widetilde y_j^{\mathrm{target}} \in \{1,\ldots,C\}. \\
&\mathbf{for}\quad k\leftarrow 0\quad\mathbf{to}\quad K_{\mathrm{rec}}-1\quad\mathbf{do} \\
&\qquad \text{Sample a synthetic mini-batch } \widetilde{\mathcal B}_k = \{ (\widetilde x_j^{(k)},\widetilde y_j^{\mathrm{target}}) \}_{j\in I_k}. \\
&\qquad \text{Sample augmentation parameters } \omega_k \sim \Omega \text{ and construct} \\
&\qquad \widetilde x_{j,k}^{\mathrm{aug}} = \mathcal A_{\omega_k} \left( \widetilde x_j^{(k)} \right), \qquad j\in I_k. \\
&\qquad \mathcal L_{\mathrm{cls}}^{(k)} = \frac{1}{|I_k|} \sum_{j\in I_k} \operatorname{CE} \left( f_{\theta_{\mathcal T}} \left( \widetilde x_{j,k}^{\mathrm{aug}} \right), \widetilde y_j^{\mathrm{target}} \right). \\
&\qquad \text{For each BN layer }l, \text{ compute the current batch statistics} \\
&\qquad \mu_l^{(k)} = \mu_l \left( \widetilde{\mathcal B}_k^{\mathrm{aug}} \right), \qquad \left(\sigma_l^2\right)^{(k)} = \sigma_l^2 \left( \widetilde{\mathcal B}_k^{\mathrm{aug}} \right). \\
&\qquad \mathcal R_{\mathrm{BN}}^{(k)} = \sum_{l=1}^{L} \left\| \mu_l^{(k)} - \mu_l^{\mathrm{run}} \right\|_2^2 + \sum_{l=1}^{L} \left\| \left(\sigma_l^2\right)^{(k)} - \left(\sigma_l^2\right)^{\mathrm{run}} \right\|_2^2. \\
&\qquad \mathcal R_{\mathrm{prior}}^{(k)} = \lambda_{\mathrm{TV}} \mathcal R_{\mathrm{TV}} \left( \widetilde{\mathcal B}_k \right) + \lambda_{\ell_2} \sum_{j\in I_k} \left\| \widetilde x_j^{(k)} \right\|_2^2. \\
&\qquad \mathcal L_{\mathrm{recover}}^{(k)} = \mathcal L_{\mathrm{cls}}^{(k)} + \lambda_{\mathrm{BN}} \mathcal R_{\mathrm{BN}}^{(k)} + \mathcal R_{\mathrm{prior}}^{(k)}. \\
&\qquad \widetilde x_j^{(k+1)} = \widetilde x_j^{(k)} - \eta_x \nabla_{\widetilde x_j^{(k)}} \mathcal L_{\mathrm{recover}}^{(k)}, \qquad j\in I_k. \\
&\qquad \text{Keep } \widetilde x_j^{(k+1)} = \widetilde x_j^{(k)} \text{ for every }j\notin I_k. \\
&\mathbf{end\ for} \\
&\text{Set the recovered synthetic images to} \qquad \widetilde{\mathcal X} = \{ \widetilde x_j \}_{j=1}^{m} = \{ \widetilde x_j^{(K_{\mathrm{rec}})} \}_{j=1}^{m}. \\[3mm]
&\mathbf{Stage\ 3:\ Relabel} \\
&\mathbf{for}\quad e\leftarrow 1\quad\mathbf{to}\quad E\quad\mathbf{do} \\
&\qquad \mathbf{for}\quad j\leftarrow 1\quad\mathbf{to}\quad m\quad\mathbf{do} \\
&\qquad\qquad \text{Sample augmentation parameters } \omega_{e,j} \sim \Omega. \\
&\qquad\qquad \widetilde x_{e,j}^{\mathrm{aug}} = \mathcal A_{\omega_{e,j}} \left( \widetilde x_j \right). \\
&\qquad\qquad z_{e,j} = f_{\theta_{\mathcal T}} \left( \widetilde x_{e,j}^{\mathrm{aug}} \right). \\
&\qquad\qquad \widetilde q_{e,j,c} = \frac{ \exp\left(z_{e,j,c}/\tau\right) }{ \displaystyle \sum_{r=1}^{C} \exp\left(z_{e,j,r}/\tau\right) }, \qquad c=1,\ldots,C. \\
&\qquad\qquad \text{Store } \left( \omega_{e,j}, \widetilde q_{e,j} \right) \text{ for the complete recovered image }\widetilde x_j. \\
&\qquad \mathbf{end\ for} \\
&\mathbf{end\ for} \\[3mm]
&\mathbf{Output:}\quad \widetilde{\mathcal D} = \left\{ \widetilde x_j, \left\{ \omega_{e,j}, \widetilde q_{e,j} \right\}_{e=1}^{E} \right\}_{j=1}^{m} \text{: relabeled synthetic dataset.}
\end{aligned}$$

# 总结

最终，SRe2L 的方法效果显著超越了 MTT，并且其完全解耦训练与优化过程的特点使得其计算量相较 MTT 大幅下降，可以直接应用在大规模数据集上。

下一章我想谈谈 RDED，这又是另一种完完全全不同的数据集蒸馏范式。与 MTT 或者 SRe2L 单纯强调数据集蒸馏效率不同，我们认为蒸馏数据集本身应该具有真实性并且包含真实图像结构。更多的，RDED 是一种直接裁剪分割原始数据集图像的方法，这意味着其拥有极低的计算负担。